In [10]:
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from scipy.sparse import hstack

DATASET_PATH = "../datasets"

df = pd.read_csv(f"{DATASET_PATH}/labeled_data/final_training_dataset_ready.csv")


# Fixed Epoch: 01/01/2026
REF_DATE = 1767225600 
ONE_YEAR = 365 * 86400

df["recency_score"] = 1 / ((REF_DATE - df["mod_time_unix"]) / ONE_YEAR + 1)
df["size_logged"] = np.log1p(df["size_bytes"])
df["name_len"] = df["filename"].apply(len)
df["path_len"] = df["path"].apply(len)
df["path_depth"] = df["path"].apply(lambda p: p.count("/") + p.count("\\"))

# Initialize HashingVectorizer
hashing_vectorizer = HashingVectorizer(
    n_features=1024,
    analyzer="char",
    norm=None,
    alternate_sign=False,
    ngram_range=(4, 4),
)

# Apply feature hashing to the 'filename' column
filename_vectors = hashing_vectorizer.fit_transform(df["filename"])
path_vectors = hashing_vectorizer.fit_transform(df["path"])

df

,filename,path,extension,size_bytes,mod_time_unix,source_user,source_type,label_finance,label_hr,label_it,label_general,recency_score,size_logged,name_len,path_len,path_depth
0,qttextlabeltest.cpp,C:\Users\sandraedwards\source\repos\avogadroli...,.cpp,6465,1.741118e+09,sandraedwards,github_real,0,0,0,0,0.547085,8.774313,19,63,7
1,nmq8332.tmp,C:\Users\jacobjones\Documents\Old_Files\Archiv...,.tmp,681574,1.536262e+09,jacobjones,govdoc_real,0,0,0,0,0.120137,13.432162,11,52,5
2,S600.wpd,C:\Users\twest\Downloads\Email_Attachments,.wpd,125829,1.766503e+09,twest,govdoc_real,0,0,0,0,0.977615,11.742687,8,42,4
3,DMC_07162001_2-01cv04_WIRELESS_V_CUMULUS.pdf,Z:\HR\Recruiting\Reqs\2024\Wireless,.pdf,31457,1.754020e+09,hansondustin,govdoc_real,0,1,0,1,0.704850,10.356409,44,35,5
4,AcceptSharedDirectoryRequest.cpp,C:\Work\Projects\aws-sdk-cpp\aws-cpp-sdk-ds\so...,.cpp,1446,1.750110e+09,bryanterin,github_real,0,0,0,0,0.648200,7.277248,32,56,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5529,file_information.rb,Z:\IT_Admin\Tools\UniDrive\app\models,.rb,107,1.671784e+09,admin,github_real,0,0,0,0,0.248360,4.682131,19,37,5
5530,ba167b99.target,C:\Users\mortonkelly\source\repos\FlyingRobotT...,.target,1747256,1.738123e+09,structural_toxic_gen,dependency_noise,0,0,0,0,0.520067,14.373558,15,112,11
5531,private.key,C:\Program Files\Enterprise Apps\ubivearound_2...,.key,2898,1.746579e+09,structural_toxic_gen,dependency_noise,0,0,0,0,0.604344,7.972121,11,100,12
5532,a614d066.pyc,C:\Users\xwilson\AppData\Local\Temp\git\draw.t...,.pyc,370322,1.753520e+09,structural_toxic_gen,dependency_noise,0,0,0,0,0.697058,12.822131,12,108,15


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from scipy.sparse import hstack

# Define target-specific valuable extensions
target_extension_map = {
    "label_finance": [".xlsx", ".xls", ".csv", ".pdf", ".accp", ".pptx"],
    "label_hr": [".docx", ".doc", ".rtf", ".pdf", ".xlsx", ".xls"],
    "label_it": [".pem",".key",".kdbx",".p12",".ovpn",".private",".wallet",".sql",".env"]
}

# Add "label_general" after defining the other labels
target_extension_map["label_general"] = list(
    set(
        target_extension_map["label_finance"]
        + target_extension_map["label_hr"]
        + target_extension_map["label_it"]
    )
)
common_dev_junk = [".pyc", ".pyi", ".lcl", ".pyd", ".map", ".ts", ".js", ".spec", ".test.js",]

# Define target-specific junk extensions
target_junk_map = {
    "label_finance": [".class", ".java", ".cpp", ".obj", ".dll", ".exe"]
    + common_dev_junk,
    "label_hr": [".class", ".java", ".py", ".obj", ".dll", ".exe"] + common_dev_junk,
    "label_it": [".tmp", ".log", ".cache"]
    + [ext for ext in common_dev_junk if ext not in [".ts", ".js"]],
}

target_junk_map["label_general"] = list(
    set(
        target_junk_map["label_finance"]
        + target_junk_map["label_hr"]
        + target_junk_map["label_it"]
    )
)

c_grid = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
weight_grid = [1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 15.0, 18.0]

models_map = {}
final_configs = {}

for target in ["label_finance", "label_hr", "label_it", "label_general"]:
# for target in ["label_it"]:

    # Prepare Data
    df_temp = df.copy()
    df_temp["valuable_ext"] = df_temp["extension"].isin(target_extension_map[target]).astype(int)
    df_temp["junk_ext"] = df_temp["extension"].isin(target_junk_map[target]).astype(int)

    df_temp.to_csv("1.csv", index=False)

    # Create feature matrix X and target vector y for the current target
    numerical_features = df_temp[
        [
            "recency_score",
            "size_logged",
            "valuable_ext",
            "junk_ext",
            "name_len",
            "path_len",
            "path_depth",
        ]
    ].values
    X_target = hstack([numerical_features, filename_vectors, path_vectors])
    y_target = df_temp[target].values

    # Split Data
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_target, y_target, df_temp.index, test_size=0.2, random_state=42, stratify=y_target
    )

    best_f1 = -1
    best_run = None

    for C in c_grid:
        for W in weight_grid:
            # Train candidate model
            model = LogisticRegression(solver="lbfgs", max_iter=1000, C=C, class_weight={0: 1, 1: W})
            model.fit(X_train, y_train)
            
            # Evaluate candidate
            y_pred = model.predict(X_test)
            p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
            
            # 3. Check University Constraints: Precision >= 0.65 AND Recall >= 0.8
            if f1 > best_f1: # We want the most balanced version of a "Success"
                best_f1 = f1
                best_run = {"C": C, "weight": W, "precision": p, "recall": r, "model": model}

    # 4. Finalize the specific target
    if best_run:
        print(f"✅ Found Winner for {target}: C={best_run['C']}, Weight={best_run['weight']}")
        print(f"📊 Results: Precision={best_run['precision']:.2f}, Recall={best_run['recall']:.2f}")
        
        models_map[target] = best_run["model"]
        final_configs[target] = {"C": best_run["C"], "weight": best_run["weight"]}
        
        # Print the final report for the best model
        print(classification_report(y_test, models_map[target].predict(X_test)))
    else:
        print(f"❌ No combination met the 0.7/0.8 threshold for {target}. Using fallback.")
        # Fallback to your previous manual target_configs values if search fails

✅ Found Winner for label_finance: C=0.8, Weight=15.0
📊 Results: Precision=0.72, Recall=0.94
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1071
           1       0.72      0.94      0.82        36

    accuracy                           0.99      1107
   macro avg       0.86      0.97      0.91      1107
weighted avg       0.99      0.99      0.99      1107

✅ Found Winner for label_hr: C=1.0, Weight=1.5
📊 Results: Precision=0.82, Recall=0.90
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1040
           1       0.82      0.90      0.86        67

    accuracy                           0.98      1107
   macro avg       0.91      0.94      0.92      1107
weighted avg       0.98      0.98      0.98      1107

✅ Found Winner for label_it: C=1.0, Weight=12.0
📊 Results: Precision=0.66, Recall=0.80
              precision    recall  f1-score   support

           0       0.99   

In [12]:
print("\ntarget_configs = {")
for key, value in final_configs.items():
    print(f'    "{key}": {{"C": {value["C"]}, "weight": {value["weight"]}}},')
print("}")


target_configs = {
    "label_finance": {"C": 0.8, "weight": 15.0},
    "label_hr": {"C": 1.0, "weight": 1.5},
    "label_it": {"C": 1.0, "weight": 12.0},
    "label_general": {"C": 0.2, "weight": 2.0},
}
